# Ray Variables
computing all variables used in computing $D$, $\Phi^2$, $\Lambda$, and $F_{\phi, \chi}$ along the each ray.
And debug them for errors

## Multiple Ray Paths

In [ ]:
from nescafe23_fluctuations import tools
import numpy as np
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

In [ ]:
latitude = 39.192499999
Omega = 7.292115e-5  # Earth Rotation Speed [rad/s]
f = 2 * Omega * np.sin(np.radians(latitude))  # [rad/s]

zeta0 = 7.3 * np.sqrt(2)
rvs = tools.compute_n23_ray_variables(f=f, zeta0=zeta0)

In [ ]:
Ds = {}

dzr = np.arange(0,500, 2)

for ray_id in ['+7','-7','+6','-6','-5b']:
    Ds[ray_id] = []
    for rv in rvs[ray_id]:
        Ds[ray_id].append(tools.compute_phase_structure_function(rv, dzr))

In [ ]:
for ray_id in ['-7']:

    for rv in rvs[ray_id]:
        plt.plot(rv['rs']/1000, rv['zs'])

plt.xlabel('range [km]')
plt.ylabel('depth [m]')
plt.ylim([-5000,0])

In [ ]:
plt.plot(rvs['-7'][0]['RT_zeta_1'])
plt.plot(rvs['-7'][1]['RT_zeta_1'])
plt.plot(rvs['-7'][1]['RT_zeta_1'])

In [ ]:
fig,axes=plt.subplots(2,3, figsize=(13,6))
axes=axes.flatten()

for k, ray_id in enumerate(['+7','-7','+6', '-6', '-5b']):
    plt.sca(axes[k])
    for D in Ds[ray_id]:
        plt.plot(dzr, np.exp(-D/2))
    plt.title(ray_id)
    plt.xlabel('$\\Delta z_r$')
    plt.grid()


plt.tight_layout()

In [ ]:
for D in Ds['+7']:
    plt.plot(dzr, np.exp(-D/2))
plt.title('+7')

In [ ]:
for D in Ds['-6']:
    plt.plot(dzr, np.exp(-D/2))
plt.title('-6')

In [ ]:
plt.imshow(np.log(rvs['-7']['H']))
plt.colorbar()

In [ ]:
plt.plot(rvs['-7']['rs']/1000, rvs['-7']['J'])
plt.xlabel('range [km]')
plt.ylabel('number of IW modes')

In [ ]:
#plt.plot(rvs['-7']['mean_j'])
plt.plot(rvs['-7']['mean_jn1'])

In [ ]:
tools.plot_ray_variables(rvs, ['+6', '-6'], 0)

In [ ]:
summands = {}

for ray_id in ['-7', '+7', '-6', '+6']:
    
    rv = rvs[ray_id]
    fc = 250
    deltaz_r = np.arange(0,500, 2)
    
    omegac = 2 * np.pi * fc
    
    ## Compute phase structure function at deltaz_r locations
    mj = (
        np.pi
        * rv["j"][None, :, None]
        * rv["Nray"].values[:, None, None]
        / rv["N0B_ray"].values[:, None, None]
    )  # (s,j,1)
    dz_s = rv["RT_zeta_1"][:, None, None] * deltaz_r[None, None, :]  # (s,1,dzr)
    summand = rv["H"].T[:, :, None] * (
        (1 - np.cos(mj * dz_s)) / rv["j"][None, :, None]
    )  # (s,j,dzr)
    # set any nan values to zero
    summand = np.where(np.isfinite(summand), summand, 0)

    summands[ray_id] = summand

In [ ]:
for ray_id in ['-7', '+7', '-6', '+6']:
    plt.plot(

In [ ]:
rvs[ray_id].keys()

In [ ]:
(rvs[ray_id]['Nray'] / rvs[ray_id]['N0B_ray']).plot()

In [ ]:
for k, ray_id in enumerate(['-7', '+7', '-6', '+6']):
    rv = rvs[ray_id]

    ## Compute phase structure function at deltaz_r locations
    mj = (
        np.pi
        * rv["j"][None, :, None]
        * rv["Nray"].values[:, None, None]
        / rv["N0B_ray"].values[:, None, None]
    )  # (s,j,1)


    dz_s = rv["RT_zeta_1"][:, None, None] * deltaz_r[None, None, :]  # (s,j,dzr)
    cos_term = (mj * dz_s)[:,:,50]

    plt.plot(np.log(np.abs(cos_term)), c=f'C{k}', alpha=0.01)

plt.hlines(np.log(2*np.pi), 0, 1000, colors='k') 

In [ ]:
for k, ray_id in enumerate(['-7', '+7', '-6', '+6']):
    _ = plt.plot(rvs[ray_id]['rs']/1000, summands[ray_id][:,:,50], alpha=0.1, c=f'C{k}')

In [ ]:
for ray_id in ['-7', '+7', '-6', '+6']:
    plt.plot(rvs[ray_id]['rs']/1000, np.sum(summands[ray_id][:,:,75], axis=1))

In [ ]:
_ = plt.plot(rv['rs']/1000, summand[:,:,75], lw=1, c='k', alpha=0.05)
plt.yscale('log')

In [ ]:
for ray_id in ['-7', '+7', '-6', '+6']:
    plt.plot(rvs[ray_id]['rs']/1000, rvs[ray_id]['zs'])
    plt.fill_between(rvs[ray_id]['rs']/1000, rvs[ray_id]['zs'] + np.sqrt(rvs[ray_id]['Rf2'])/2, rvs[ray_id]['zs'] - np.sqrt(rvs[ray_id]['Rf2'])/2, alpha=0.3)
    

In [ ]:
D = {}
Lamda = {}
Phi2 = {}
F_phi = {}
F_chi = {}

dzr = np.arange(0,500, 2)
m = 2 * np.pi * np.logspace(-5, 0, 500)[None, :]  # [rad / m]

for ray_id in ['-6', '+6', '-7', '+7']:
    D[ray_id] = tools.compute_phase_structure_function(rvs[ray_id], dzr)
    Lamda[ray_id], Phi2[ray_id] = res = tools.compute_lambda_phi(rvs[ray_id])
    F_phi[ray_id], F_chi[ray_id] = tools.compute_F_phi_chi(rvs[ray_id], m)

In [ ]:
plt.scatter(list(Lamda.values()), np.sqrt(list(Phi2.values())))
plt.yscale('log')
plt.xscale('log')

plt.ylim([1e0, 1e2])
plt.xlim([1e-2, 1e0])

plt.grid()

In [ ]:
for ray_id in ['-7', '+7', '-6', '+6']:
    plt.plot(dzr, np.exp(-D[ray_id]/2), label=ray_id)

plt.legend()

In [ ]:
fig = plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
for ray_id in D:
    plt.plot(m[0]/(2*np.pi), F_phi[ray_id], label=ray_id)

plt.legend()


plt.yscale('log')
plt.xscale('log')

plt.subplot(1,2,2)
for ray_id in D:
    plt.plot(m[0]/(2*np.pi), F_chi[ray_id], label=ray_id)

plt.legend()


plt.yscale('log')
plt.xscale('log')

### investigate ray seperation

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
import pickle
import matplotlib.pyplot as plt

_ = load_dotenv(find_dotenv())

In [ ]:
fn = f"{os.environ['NESCAFE23_DATADIR']}dataproducts/ray_run_forward_back.pkl"

with open(fn, 'rb') as f:
    ray_run = pickle.load(f)

rays = ray_run['rays']
eigs = ray_run['eigs']

In [ ]:
ray_seperation = rays.zs[1:,:] - rays.zs[:-1,:]

In [ ]:
%matplotlib widget

In [ ]:
plt.close()
plt.imshow(np.log(np.abs(ray_seperation)), aspect='auto', extent = (rays.rs[0,0], rays.rs[0,-1], rays.thetas[0], rays.thetas[-1]))
plt.colorbar()

plt.hlines(eigs.received_angles[0], 0, 150000, lw=1, colors='k', alpha=0.3)

In [ ]:
eig_idx = np.where(eigs.ray_id[0] == '7.0')[0]

In [ ]:
eig_idx

In [ ]:
plt.close()
plt.plot(eigs.rs[0][eig_idx,:].T, eigs.zs[0][eig_idx,:].T)

In [ ]:
eigs.received_angles

## Investigate Vertical Coherence

In [ ]:
from nescafe23_fluctuations import tools
import numpy as np
import matplotlib.pyplot as plt
from dotenv import load_dotenv, find_dotenv
import os
import pickle

load_dotenv(find_dotenv())

%load_ext autoreload
%autoreload 2

In [ ]:
latitude = 39.192499999
Omega = 7.292115e-5  # Earth Rotation Speed [rad/s]
f = 2 * Omega * np.sin(np.radians(latitude))  # [rad/s]

zeta0 = 7.3 * np.sqrt(2)
rvs = tools.compute_n23_ray_variables(f=f, zeta0=zeta0)

In [ ]:
tools.plot_ray_variables(rvs, ['+6', '-6'])

In [ ]:
fn = f"{os.environ['NESCAFE23_DATADIR']}dataproducts/ray_run_forward_back.pkl"

with open(fn, 'rb') as f:
    ray_run = pickle.load(f)

In [ ]:
eig_idx_forward = np.where(ray_run['eigs'].ray_id[0] == '6.0')[0][0]
eig_idx_backward = np.where(ray_run['eigs_back'].ray_id[0] == '6.0')[0][0]

In [ ]:
plt.plot(rvs['+6'][0]['rs'], rvs['+6'][0]['zs'])
plt.plot(rvs['+6'][0]['rs_back'], np.flip(rvs['+6'][0]['zs_back']))

In [ ]:
dx1 = np.gradient(rvs['+6'][0]['RT_zeta_1'], rvs['+6'][0]['rs'])
dx2 = np.gradient(rvs['+6'][0]['RT_zeta_2'], rvs['+6'][0]['rs'])

In [ ]:
plt.subplot(1,2,1)
plt.plot(rvs['+6'][0]['rs']/1000, (rvs['+6'][0]['RT_zeta_1'] * dx2) - (rvs['+6'][0]['RT_zeta_2'] * dx1))

ax2 = plt.twinx()
plt.plot(rvs['+6'][0]['rs']/1000, (rvs['+6'][0]['RT_zeta_1'] * rvs['+6'][0]['RT_zeta_2'] ), c='C1')

plt.subplot(1,2,2)
plt.plot(rvs['+6'][0]['rs']/1000, (rvs['+6'][0]['Rf2']))

plt.tight_layout()

In [ ]:
ray_id = '-7'

plt.plot(rvs[ray_id][0]['RT_zeta_1'])
plt.plot(rvs[ray_id][0]['RT_zeta_2'])

plt.plot(rvs[ray_id][0]['RT_zeta_1_2'])
plt.plot(rvs[ray_id][0]['RT_zeta_2_2'])

In [ ]:
plt.close()

plt.plot(ray_run['envi'].sound_speed.sel({'range':30000}, method='nearest').differentiate('depth'), ray_run['envi'].sound_speed.depth)


plt.hlines(250, -1,1)

plt.ylim([1000,0])

In [ ]:
rvs['+6'][0]['zs'].max()

In [ ]:
ray_run['envi'].sound_speed.differentiate('depth').plot()


plt.plot(rvs['+6'][0]['rs'], -rvs['+6'][0]['zs'])
#plt.plot(rvs['-6'][0]['rs'], -rvs['-6'][0]['zs'])

plt.ylim([400,200])

In [ ]:
fn = f"{os.environ['NESCAFE23_DATADIR']}dataproducts/ray_run_forward_back.pkl"

with open(fn, 'rb') as f:
    ray_run = pickle.load(f)

In [ ]:
ray_run['rays'].plot_time_front()
plt.xlim([100,102])

In [ ]:
tools.plot_ray_variables(rvs, ['+6'], 0)
tools.plot_ray_variables(rvs, ['+6'], 1)

In [ ]:
tools.plot_ray_variables(rvs, ['+6','-6'], 0)